# 01 — Bronze Layer: Raw Data Ingestion

This notebook demonstrates the **Bronze layer** of the B3 Data Platform.

**Goal:** ingest raw daily OHLCV prices from Yahoo Finance and persist them as-is
(no transformation) into the Bronze Parquet store.

Architecture reminder:
```
Yahoo Finance API  ──►  Bronze (raw Parquet)  ──►  Silver  ──►  Gold
```
**Rules for Bronze:**
- Data is written exactly as received — no value changes.
- Only metadata columns are added: `source` and `ingested_at`.
- Partitioned by `trade_date` for fast downstream reads.

In [1]:
# Configuração básica do ambiente do notebook.
# - `sys.path.insert(0, "..")` permite importar os módulos do projeto
#   (a_configs, c_ingestion, d_processing, ...) a partir do diretório raiz.
# - Polars é a engine principal de manipulação de dados (camadas Bronze/Silver).
import sys
sys.path.insert(0, "..")

from datetime import date, timedelta

import polars as pl

# Exibe até 20 linhas por DataFrame para facilitar a inspeção visual.
pl.Config.set_tbl_rows(20)
print(f"Polars version: {pl.__version__}")

Polars version: 1.42.0


## 1. Fetch raw prices from Yahoo Finance

In [2]:
# 1) Extração: baixa os preços diários (OHLCV) direto do Yahoo Finance.
#    Este é o ÚNICO ponto onde o notebook fala com a fonte externa.
from c_ingestion.yahoo_finance import fetch_daily_prices

# Janela de 3 meses encerrando hoje — suficiente para demonstração.
END   = date.today()
START = END - timedelta(days=90)

# Subconjunto reduzido (5 ativos) usado apenas no exemplo interativo.
# O pipeline completo (célula final) usa a lista DEFAULT_TICKERS de a_configs.
DEMO_TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

raw_df = fetch_daily_prices(tickers=DEMO_TICKERS, start=START, end=END)
print(f"Fetched {len(raw_df):,} rows for {raw_df['ticker'].n_unique()} tickers")
raw_df.head(10)

{"timestamp": "2026-06-26T01:23:17.509638+00:00", "level": "INFO", "logger": "c_ingestion.yahoo_finance", "message": "Starting Yahoo Finance ingestion", "module": "yahoo_finance", "func": "fetch_daily_prices", "line": 96, "name": "c_ingestion.yahoo_finance", "msg": "Starting Yahoo Finance ingestion", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\c_ingestion\\yahoo_finance.py", "filename": "yahoo_finance.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 96, "funcName": "fetch_daily_prices", "created": 1782436997.509584, "msecs": 509.0, "relativeCreated": 177047.2685, "thread": 11904, "threadName": "MainThread", "processName": "MainProcess", "process": 24816, "taskName": "Task-77", "tickers": 5, "start": "2026-03-27", "end": "2026-06-25"}
{"timestamp": "2026-06-26T01:23:31.021816+00:00", "level": "INFO", "logger": "c_ingestion.yahoo_finance", "message": "Inges

ticker,trade_date,open,high,low,close,adj_close,volume
str,date,f64,f64,f64,f64,f64,i64
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900
"""PETR4.SA""",2026-03-30,49.75,50.689999,49.490002,49.669998,48.168758,51454300
"""PETR4.SA""",2026-03-31,50.07,50.549999,47.650002,48.669998,47.198982,78931700
"""PETR4.SA""",2026-04-01,47.700001,48.07,46.77,47.369999,45.938274,66488400
"""PETR4.SA""",2026-04-02,49.299999,49.459999,47.950001,48.150002,46.694702,38536900
"""PETR4.SA""",2026-04-06,47.990002,49.0,47.880001,48.939999,47.460823,27848100
"""PETR4.SA""",2026-04-07,49.09,49.560001,48.400002,48.509998,47.043819,33362800
"""PETR4.SA""",2026-04-08,44.700001,46.639999,44.529999,46.610001,45.201248,87580500
"""PETR4.SA""",2026-04-09,47.540001,48.549999,47.029999,47.900002,46.452259,58206500


## 2. Inspect raw data

In [3]:
# 2a) Inspeção rápida: schema, contagem de nulos e estatística descritiva.
# Útil para validar que o download retornou dados consistentes antes de gravar.
print("Schema:")
print(raw_df.schema)
print("\nNull counts:")
print(raw_df.null_count())
print("\nBasic stats:")
raw_df.describe()

Schema:
Schema({'ticker': String, 'trade_date': Date, 'open': Float64, 'high': Float64, 'low': Float64, 'close': Float64, 'adj_close': Float64, 'volume': Int64})

Null counts:
shape: (1, 8)
┌────────┬────────────┬──────┬──────┬─────┬───────┬───────────┬────────┐
│ ticker ┆ trade_date ┆ open ┆ high ┆ low ┆ close ┆ adj_close ┆ volume │
│ ---    ┆ ---        ┆ ---  ┆ ---  ┆ --- ┆ ---   ┆ ---       ┆ ---    │
│ u32    ┆ u32        ┆ u32  ┆ u32  ┆ u32 ┆ u32   ┆ u32       ┆ u32    │
╞════════╪════════════╪══════╪══════╪═════╪═══════╪═══════════╪════════╡
│ 0      ┆ 0          ┆ 0    ┆ 0    ┆ 0   ┆ 0     ┆ 0         ┆ 0      │
└────────┴────────────┴──────┴──────┴─────┴───────┴───────────┴────────┘

Basic stats:


statistic,ticker,trade_date,open,high,low,close,adj_close,volume
str,str,str,f64,f64,f64,f64,f64,f64
"""count""","""300""","""300""",300.0,300.0,300.0,300.0,300.0,300.0
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,"""2026-05-11 15:12:00""",40.785267,41.224933,40.3246,40.7627,40.518635,3.1334303e7
"""std""",null,null,24.138005,24.394588,23.854262,24.149731,24.12708,1.6411e7
"""min""","""ABEV3.SA""","""2026-03-27""",14.36,14.56,14.32,14.33,14.290209,7.0595e6
"""25%""",null,"""2026-04-20""",17.620001,17.93,17.5,17.66,17.66,2.01966e7
"""50%""",null,"""2026-05-13""",40.290001,40.650002,40.099998,40.400002,40.020744,2.67209e7
"""75%""",null,"""2026-06-02""",47.639999,48.169998,47.169998,47.77,46.586304,3.70226e7
"""max""","""VALE3.SA""","""2026-06-24""",88.889999,89.75,88.029999,89.75,89.75,1.099523e8


In [4]:
# 2b) Cobertura temporal por ativo.
# Conferimos primeira/última data e quantidade de pregões — qualquer ativo
# com poucos pregões pode indicar problema no download ou ticker suspenso.
raw_df.group_by("ticker").agg(
    pl.col("trade_date").min().alias("first_date"),
    pl.col("trade_date").max().alias("last_date"),
    pl.col("trade_date").n_unique().alias("trading_days"),
).sort("ticker")

ticker,first_date,last_date,trading_days
str,date,date,u32
"""ABEV3.SA""",2026-03-27,2026-06-24,60
"""BBDC4.SA""",2026-03-27,2026-06-24,60
"""ITUB4.SA""",2026-03-27,2026-06-24,60
"""PETR4.SA""",2026-03-27,2026-06-24,60
"""VALE3.SA""",2026-03-27,2026-06-24,60


## 3. Write to Bronze layer

In [5]:
# 3) Carga na camada Bronze.
# Grava os dados como Parquet particionado por trade_date.
# A camada Bronze é IMUTÁVEL: não alteramos valores, apenas adicionamos
# colunas de metadata (`source` e `ingested_at`) para rastreabilidade.
from d_processing.bronze.ingest import write_bronze

bronze_path = write_bronze(raw_df, source="yahoo_finance")
print(f"Written to: {bronze_path}")

{"timestamp": "2026-06-26T01:23:53.681755+00:00", "level": "INFO", "logger": "d_processing.bronze.ingest", "message": "Bronze write complete", "module": "ingest", "func": "write_bronze", "line": 62, "name": "d_processing.bronze.ingest", "msg": "Bronze write complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\bronze\\ingest.py", "filename": "ingest.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 62, "funcName": "write_bronze", "created": 1782437033.681695, "msecs": 681.0, "relativeCreated": 213219.3798, "thread": 11904, "threadName": "MainThread", "processName": "MainProcess", "process": 24816, "taskName": "Task-122", "source": "yahoo_finance", "rows": 300, "partitions": 60, "output": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\j_data\\bronze\\yahoo_finance", "ingested_at": "2026-06-26T01:23:49.041495+00:

## 4. Read back from Bronze and verify

In [6]:
# 4) Releitura: garante que o write foi consistente e mostra as colunas
# de metadata acrescentadas pela camada Bronze.
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}")
print(f"Columns added: {[c for c in bronze_df.columns if c not in raw_df.columns]}")
bronze_df.head(10)

{"timestamp": "2026-06-26T01:24:00.741532+00:00", "level": "INFO", "logger": "d_processing.bronze.ingest", "message": "Bronze read complete", "module": "ingest", "func": "read_bronze", "line": 107, "name": "d_processing.bronze.ingest", "msg": "Bronze read complete", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\d_processing\\bronze\\ingest.py", "filename": "ingest.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 107, "funcName": "read_bronze", "created": 1782437040.7414386, "msecs": 741.0, "relativeCreated": 220279.1233, "thread": 11904, "threadName": "MainThread", "processName": "MainProcess", "process": 24816, "taskName": "Task-137", "rows": 300, "source": "yahoo_finance"}
Bronze rows: 300
Columns added: ['source', 'ingested_at']


ticker,trade_date,open,high,low,close,adj_close,volume,source,ingested_at
str,date,f64,f64,f64,f64,f64,i64,str,"datetime[μs, UTC]"
"""PETR4.SA""",2026-03-27,48.5,49.439999,48.130001,49.41,47.916618,54170900,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""VALE3.SA""",2026-03-27,78.470001,79.919998,78.169998,79.0,79.0,11101700,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""ITUB4.SA""",2026-03-27,41.810001,41.810001,41.18,41.450001,41.026436,27910700,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""BBDC4.SA""",2026-03-27,18.709999,18.809999,18.440001,18.52,18.1793,20727200,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""ABEV3.SA""",2026-03-27,14.79,15.0,14.74,14.76,14.719015,17884700,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""PETR4.SA""",2026-03-30,49.75,50.689999,49.490002,49.669998,48.168758,51454300,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""VALE3.SA""",2026-03-30,80.25,81.059998,79.290001,79.5,79.5,18776700,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""ITUB4.SA""",2026-03-30,42.0,42.060001,41.23,41.599998,41.1749,18528100,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC
"""BBDC4.SA""",2026-03-30,18.77,18.790001,18.379999,18.469999,18.130219,32459400,"""yahoo_finance""",2026-06-26 01:23:49.041495 UTC


## 5. Quick visualization — closing prices

In [10]:
# 5) Visualização: preços de fechamento por ativo ao longo do tempo.
# Usamos Plotly Express para ter um gráfico interativo (zoom, hover, legenda).
# Template `plotly_white` melhora a leitura em apresentações/projeção.
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

# Plotly recebe pandas, por isso o `.to_pandas()` no final do encadeamento.
fig = px.line(
    bronze_df.sort("trade_date").to_pandas(),
    x="trade_date",
    y="close",
    color="ticker",
    title="Preços de Fechamento das Ações da B3 — Camada Bronze (dados brutos)",
    labels={
        "trade_date": "Data da Negociação",
        "close": "Preço de Fechamento (R$)",
        "ticker": "Ativo",
    },
)
# Ajustes de layout pensados em legibilidade para apresentação.
fig.update_layout(
    height=500,
    hovermode="x unified",                       # tooltip único para todos os ativos
    legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
    margin=dict(l=60, r=40, t=70, b=50),
)
fig.update_yaxes(tickprefix="R$ ", separatethousands=True)
fig.update_xaxes(tickformat="%d/%m/%Y")
fig.show()

## 6. Run full Bronze pipeline (all default tickers)

In [8]:
# 6) Execução do pipeline completo da camada Bronze.
# Mesmo fluxo extract→load das células acima, mas encapsulado numa classe
# (BronzePipeline) que é a unidade chamada pelas DAGs do Airflow.
from f_pipelines.bronze_pipeline import BronzePipeline, BronzePipelineConfig

cfg = BronzePipelineConfig(start=START, end=END)
pipeline = BronzePipeline(config=cfg)
result = pipeline.run()
print(f"Pipeline produced {len(result):,} rows")

{"timestamp": "2026-06-26T01:29:24.644241+00:00", "level": "INFO", "logger": "f_pipelines.bronze_pipeline", "message": "BronzePipeline started", "module": "bronze_pipeline", "func": "run", "line": 59, "name": "f_pipelines.bronze_pipeline", "msg": "BronzePipeline started", "args": [], "levelname": "INFO", "levelno": 20, "pathname": "\\\\wsl.localhost\\ubuntu-llm\\home\\efcardoso\\projects\\b3-data-plataform\\i_notebooks\\..\\f_pipelines\\bronze_pipeline.py", "filename": "bronze_pipeline.py", "exc_info": null, "exc_text": null, "stack_info": null, "lineno": 59, "funcName": "run", "created": 1782437364.6442106, "msecs": 644.0, "relativeCreated": 544181.8954, "thread": 11904, "threadName": "MainThread", "processName": "MainProcess", "process": 24816, "taskName": "Task-158"}
{"timestamp": "2026-06-26T01:29:24.645252+00:00", "level": "INFO", "logger": "f_pipelines.bronze_pipeline", "message": "Bronze extraction started", "module": "bronze_pipeline", "func": "extract", "line": 41, "name": "f_